# FDM expansion for Diffusive Equations

In [305]:
import warnings
warnings.filterwarnings("ignore")
from pylab import *
import sympy as sp
import numpy as np
import re as regex
from sympy.tensor.indexed import Indexed
from IPython.display import display, Math, Latex
%run ~/Desktop/Bazarin-R.L.M.-AcademicDocumentation/docs/source/modules/eq_funcs.ipynb

us = sp.symbols('u_{x}^{t}')
omega = sp.symbols(r'\omega')
omg1=1-omega
cs = sp.symbols('c_s')

 ## One-Dimensional Cases: $D1Qq$ Lattices

In [306]:
import sympy as sp
ul = {}
past_steps = [1, 2, 3, 4, 5, 6,]
space_offsets = [1, 2, 3, 4, 5, 6]
# time_map: key_prefix -> (latex_expr): "+1" -> u^{t+1},  "0" -> u^{t},  "-1".."-4" -> u^{t-k}
time_map = {"+1": "t+1","0":  "t",**{f"-{k}": f"t-{k}" for k in past_steps},}

# Non-shifted symbols
for tk, te in time_map.items():
    ul[f"ut{tk}"] = sp.Symbol(fr'u^{{{te}}}_{{x}}')

# Shifted symbols (skip the future step "+1")
for tk, te in time_map.items():
    if tk == "+1":
        continue
    for ssign in ("+", "-"):
        for d in space_offsets:
            ul[f"ut{tk}d{ssign}{d}"] = sp.Symbol(fr'u^{{{te}}}_{{x{ssign}{d}}}')

In [347]:
# ul

In [308]:
def make_shift_symbols(prefix=r'T_{\Delta_{x}}', order=9, var='c', start=0):
    names = [f'{prefix}^{{{var}_{{{i}}}}}' for i in range(start, start + order)]
    return sp.symbols(names)
Txc = make_shift_symbols(order=9, var='c')   # list of SymPy symbols
# Txc

In [266]:
from itertools import combinations, combinations_with_replacement
# ----Multi-index shift symbols with permutation deduplication------
def make_perm_shift_symbols(prefix=r'T_{\Delta_{x}}',
                            order=9,
                            var='c',
                            start=0,
                            length=2,
                            include_repeats=False,
                            canonical=True):
    indices = range(start, start + order)
    if include_repeats:
        combos = combinations_with_replacement(indices, length)
    else:
        combos = combinations(indices, length)
    out = {}
    for combo in combos:
        key = tuple(sorted(combo)) if canonical else tuple(combo)
        sup = ''.join(f'{var}_{{{i}}}' for i in key)
        out[key] = sp.Symbol(fr'{prefix}^{{{sup}}}')
    return out

In [309]:
p2T = make_perm_shift_symbols(order=7, length=2, include_repeats=False)
# print(p2T[(0, 1)])   # T_{\Delta_{x}}^{c_{0}c_{1}}
print(len(p2T))      # 36

21


In [310]:
p3T = make_perm_shift_symbols(order=7, length=3, include_repeats=False)
# print(p2T[(0,1,2)])   # T_{\Delta_{x}}^{c_{0}c_{1}}
print(len(p3T))      # 36

35


In [311]:
def wheigt_symbols(prefix=r'w', order=9, start=0):
    names = [f'{prefix}_{{{i}}}' for i in range(start, start + order)]
    return sp.symbols(names)
w = wheigt_symbols(order=9)   # list of SymPy symbols
# w[0]

### $D1Q3$

In [270]:
import warnings
warnings.filterwarnings("ignore")
from pylab import *
from sympy import *
import numpy as np
q=3
cx=np.array([0,1,-1])
wis=np.array([w[0],w[1],w[2]])
feq = us*wis

In [271]:
# OrderT = Matrix([0, 0, 0])
# # First order (single index)
# for i0 in range(q):
#     OrderT[0] = OrderT[0] + omega * feq[i0] * Txc[i0] + (1 - omega) * Txc[i0] * us
#     # Second order (unordered pairs i0 < i1)
#     for i1 in range(i0 + 1, q):
#         OrderT[1] = OrderT[1] + omega*(feq[i1]+feq[i0]) * Txc[i0]*Txc[i1] + (1-omega)*Txc[i0]*Txc[i1]*us
#         # Third order (unordered triples i0 < i1 < i2)
#         for i2 in range(i1 + 1, q):
#             OrderT[2] = OrderT[2] + omega*(feq[i2]+feq[i1]+feq[i0]) *Txc[i0]*Txc[i1]*Txc[i2] + (1-omega)*Txc[i0]*Txc[i1]*Txc[i2]*us

In [278]:
from itertools import combinations
import sympy as sp

OrderT = sp.Matrix([
    sum(
        (-omg1)**(n-1)
        * sp.prod(Txc[i] for i in idx)
        * (omega * sum(feq[i] for i in idx) + (1-omega)*us)
        for idx in combinations(range(q), n)
    )
    for n in range(1, 4)
])

In [279]:
# display(OrderT[0])
mt0=sp.collect(sp.expand(OrderT[0])
           .subs({Txc[1]*Txc[2]:1,Txc[0]:1})
           .subs(Txc[1]*us,ul['ut0d-1'])
           .subs(Txc[2]*us,ul['ut0d+1'])
           .subs({w[2]:Rational(1,6),w[1]:Rational(1,6),w[0]:Rational(2,3)})
           ,{us,ul['ut0d-1'],ul['ut0d+1']}
          )
display(mt0)

u^{t}_{x+1}*(1 - 5*\omega/6) + u^{t}_{x-1}*(1 - 5*\omega/6) + u_{x}^{t}*(1 - \omega/3)

In [280]:
# display(OrderT[1])
mt1=sp.collect(sp.expand(OrderT[1])
           .subs({Txc[1]*Txc[2]:1,Txc[0]:1})
           .subs(Txc[1]*us,ul['ut-1d-1'])
           .subs(Txc[2]*us,ul['ut-1d+1'])
           .subs(us,ul['ut-1'])
           .subs({w[2]:Rational(1,6),w[1]:Rational(1,6),w[0]:Rational(2,3)})
           ,{ul['ut-1'],ul['ut-1d-1'],ul['ut-1d+1']}
          )
display(mt1)

u^{t-1}_{x+1}*(-\omega**2/6 + 7*\omega/6 - 1) + u^{t-1}_{x-1}*(-\omega**2/6 + 7*\omega/6 - 1) + u^{t-1}_{x}*(-2*\omega**2/3 + 5*\omega/3 - 1)

In [281]:
mt2=sp.simplify(sp.expand(OrderT[2])
           .subs({Txc[1]*Txc[2]:1,Txc[0]:1})
           .subs({w[1]:Rational(1,6),w[2]:Rational(1,6),w[0]:Rational(2,3)})
           .subs(us,ul['ut-2'])
          )
display(mt2)

u^{t-2}_{x}*(\omega**2 - 2*\omega + 1)

In [283]:
# sp.Eq(ul['ut+1'],mt0+mt1+mt2)
display(Math(fdm_ordered_brokeq(sp.Eq(ul['ut+1'],mt0+mt1+mt2), terms_per_line=3, symbols=ul)))

<IPython.core.display.Math object>

### $D1Q5ZOT$

In [284]:
import warnings
warnings.filterwarnings("ignore")
from pylab import *
from sympy import *
import numpy as np
q=5
cx=np.array([0,1,-1,3,-3])
wis=np.array([w[0],w[1],w[2],w[3],w[4]])
w0 = sp.Rational(16, 45) + sp.Rational(4, 45)*sp.sqrt(10)
w1 = sp.Rational(3, 10) - sp.Rational(3, 80)*sp.sqrt(10)
w2 = sp.Rational(1, 45) - sp.Rational(1, 144)*sp.sqrt(10)
feq = us*wis

In [285]:
# OrderT = Matrix([0, 0, 0, 0, 0])
# # First order (single index)
# for i0 in range(q):
#     OrderT[0] = OrderT[0] + (-omg1)**(0)*(omega*feq[i0]*Txc[i0] + (1-omega)*Txc[i0]*us)
#     # Second order (unordered pairs i0 < i1)
#     for i1 in range(i0 + 1, q):
#         OrderT[1] = OrderT[1] + (-omg1)**(1)*(omega*(feq[i1]+feq[i0]) * Txc[i0]*Txc[i1] + (1-omega)*Txc[i0]*Txc[i1]*us)
#         # Third order (unordered triples i0 < i1 < i2)
#         for i2 in range(i1 + 1, q):
#             OrderT[2] = OrderT[2] + (-omg1)**(2)*(omega*(feq[i2]+feq[i1]+feq[i0]) *Txc[i0]*Txc[i1]*Txc[i2] + (1-omega)*Txc[i0]*Txc[i1]*Txc[i2]*us)
#             # Fourth order (unordered triples i0 < i1 < i2 < i3)
#             for i3 in range(i2 + 1, q):
#                 OrderT[3] = OrderT[3] + (-omg1)**(3)*(omega*(feq[i3]+feq[i2]+feq[i1]+feq[i0]) *Txc[i0]*Txc[i1]*Txc[i2]*Txc[i3] 
#                                          + (1-omega)*Txc[i0]*Txc[i1]*Txc[i2]*Txc[i3]*us)
#                 # Fiveth order (unordered triples i0 < i1 < i2 < i3 < i4)
#                 for i4 in range(i3 + 1, q):
#                     OrderT[4] = OrderT[4] + (-omg1)**(4)*(omega*(feq[i4]+feq[i3]+feq[i2]+feq[i1]+feq[i0]) *Txc[i0]*Txc[i1]*Txc[i2]*Txc[i3]*Txc[i4] 
#                                          + (1-omega)*Txc[i0]*Txc[i1]*Txc[i2]*Txc[i3]*Txc[i4]*us)

In [286]:
from itertools import combinations
import sympy as sp

OrderT = sp.Matrix([
    sum(
        (-omg1)**(n-1)
        * sp.prod(Txc[i] for i in idx)
        * (omega * sum(feq[i] for i in idx) + (1-omega)*us)
        for idx in combinations(range(q), n)
    )
    for n in range(1, 6)
])

In [287]:
# display(OrderT[0])
mt0=sp.collect(OrderT[0]
           .subs({Txc[1]*Txc[2]:1,Txc[0]:1})
           .subs(Txc[1]*us,ul['ut0d-1'])
           .subs(Txc[2]*us,ul['ut0d+1'])
           .subs(Txc[3]*us,ul['ut0d-3'])
           .subs(Txc[4]*us,ul['ut0d+3'])
           # .subs({w[2]:Rational(1,6),w[1]:Rational(1,6),w[0]:Rational(2,3)})
           ,{us,ul['ut0d-1'],ul['ut0d+1'],ul['ut0d-3'],ul['ut0d+3']}
          )
display(mt0)

u_{x}^{t}*(T_{\Delta_{x}}^{c_{1}}*(\omega*w_{1} - \omega + 1) + T_{\Delta_{x}}^{c_{2}}*(\omega*w_{2} - \omega + 1) + T_{\Delta_{x}}^{c_{3}}*(\omega*w_{3} - \omega + 1) + T_{\Delta_{x}}^{c_{4}}*(\omega*w_{4} - \omega + 1) + \omega*w_{0} - \omega + 1)

In [288]:
# display(OrderT[1])
mt1=sp.collect(sp.expand(OrderT[1])
           .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[0]:1,
                  Txc[1]*Txc[3]:p2T[(1, 3)], Txc[2]*Txc[4]:p2T[(2, 4)], Txc[1]*Txc[4]:p2T[(1, 4)], Txc[2]*Txc[3]:p2T[(2, 3)]})
           .subs(Txc[1]*us,ul['ut-1d-1'])
           .subs(Txc[2]*us,ul['ut-1d+1'])
           .subs(Txc[3]*us,ul['ut-1d-3'])
           .subs(Txc[4]*us,ul['ut-1d+3'])        
           .subs(p2T[(1, 3)]*us,ul['ut-1d-4'])
           .subs(p2T[(2, 4)]*us,ul['ut-1d+4'])
           .subs(p2T[(1, 4)]*us,ul['ut-1d+2'])
           .subs(p2T[(2, 3)]*us,ul['ut-1d-2'])
           .subs(us,ul['ut-1'])
           # .subs({w[2]:Rational(1,6),w[1]:Rational(1,6),w[0]:Rational(2,3)})
           ,{ul['ut-1'],ul['ut-1d-1'],ul['ut-1d+1'],ul['ut-1d-2'],ul['ut-1d+2'],ul['ut-1d-3'],ul['ut-1d+3'],ul['ut-1d-4'],ul['ut-1d+4']}
          )
# display(mt1)
brokeq2(mt1, terms_per_line=1)

<IPython.core.display.Math object>

In [289]:
# display(OrderT[2])
mt2=sp.collect(sp.expand(OrderT[2])
           .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[0]:1})
           .subs({Txc[1]*Txc[3]:p2T[(1, 3)], Txc[2]*Txc[4]:p2T[(2, 4)], Txc[1]*Txc[4]:p2T[(1, 4)], Txc[2]*Txc[3]:p2T[(2, 3)]})
           .subs(Txc[1]*us,ul['ut-2d-1'])
           .subs(Txc[2]*us,ul['ut-2d+1'])
           .subs(Txc[3]*us,ul['ut-2d-3'])
           .subs(Txc[4]*us,ul['ut-2d+3'])        
           .subs(p2T[(1, 3)]*us,ul['ut-2d-4'])
           .subs(p2T[(2, 4)]*us,ul['ut-2d+4'])
           .subs(p2T[(1, 4)]*us,ul['ut-2d+2'])
           .subs(p2T[(2, 3)]*us,ul['ut-2d-2'])
           .subs(us,ul['ut-2'])
           # .subs({w[2]:Rational(1,6),w[1]:Rational(1,6),w[0]:Rational(2,3)})
           ,{ul['ut-2'],ul['ut-2d-1'],ul['ut-2d+1'],ul['ut-2d-2'],ul['ut-2d+2'],ul['ut-2d-3'],ul['ut-2d+3'],ul['ut-2d-4'],ul['ut-2d+4']}
          )
# display(mt2)
brokeq2(mt2, terms_per_line=1)

<IPython.core.display.Math object>

In [290]:
# display(OrderT[3])
mt3=sp.collect(sp.expand(OrderT[3])
           .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[0]:1})
           .subs({Txc[1]*Txc[3]:p2T[(1, 3)], Txc[2]*Txc[4]:p2T[(2, 4)], Txc[1]*Txc[4]:p2T[(1, 4)], Txc[2]*Txc[3]:p2T[(2, 3)]})
           .subs(Txc[1]*us,ul['ut-3d-1'])
           .subs(Txc[2]*us,ul['ut-3d+1'])
           .subs(Txc[3]*us,ul['ut-3d-3'])
           .subs(Txc[4]*us,ul['ut-3d+3'])        
           .subs(p2T[(1, 3)]*us,ul['ut-3d-4'])
           .subs(p2T[(2, 4)]*us,ul['ut-3d+4'])
           .subs(p2T[(1, 4)]*us,ul['ut-3d+2'])
           .subs(p2T[(2, 3)]*us,ul['ut-3d-2'])
           .subs(us,ul['ut-3'])
           # .subs({w[2]:Rational(1,6),w[1]:Rational(1,6),w[0]:Rational(2,3)})
           ,{ul['ut-3'],ul['ut-3d-1'],ul['ut-3d+1'],ul['ut-3d-2'],ul['ut-3d+2'],ul['ut-3d-3'],ul['ut-3d+3'],ul['ut-3d-4'],ul['ut-3d+4']}
          )
# display(mt3)
brokeq2(mt3, terms_per_line=1)

<IPython.core.display.Math object>

In [291]:
# display(OrderT[4])
mt4=sp.collect(sp.expand(OrderT[4])
           .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[0]:1})
           .subs({Txc[1]*Txc[3]:p2T[(1, 3)], Txc[2]*Txc[4]:p2T[(2, 4)], Txc[1]*Txc[4]:p2T[(1, 4)], Txc[2]*Txc[3]:p2T[(2, 3)]})
           .subs(Txc[1]*us,ul['ut-4d-1'])
           .subs(Txc[2]*us,ul['ut-4d+1'])
           .subs(Txc[3]*us,ul['ut-4d-3'])
           .subs(Txc[4]*us,ul['ut-4d+3'])        
           .subs(p2T[(1, 3)]*us,ul['ut-4d-4'])
           .subs(p2T[(2, 4)]*us,ul['ut-4d+4'])
           .subs(p2T[(1, 4)]*us,ul['ut-4d+2'])
           .subs(p2T[(2, 3)]*us,ul['ut-4d-2'])
           .subs(us,ul['ut-4'])
           # .subs({w[2]:Rational(1,6),w[1]:Rational(1,6),w[0]:Rational(2,3)})
           ,{ul['ut-4'],ul['ut-4d-1'],ul['ut-4d+1'],ul['ut-4d-2'],ul['ut-4d+2'],ul['ut-4d-3'],ul['ut-4d+3'],ul['ut-4d-4'],ul['ut-4d+4']}
          )
# display(mt4)
brokeq2(mt4, terms_per_line=1)

<IPython.core.display.Math object>

In [292]:
# display_broken_equation(sp.Eq(ul['ut+1'],mt0+mt1+mt2+mt3+mt4), terms_per_line=1)
display(Math(fdm_ordered_brokeq(sp.Eq(ul['ut+1'],mt0+mt1+mt2+mt3+mt4), terms_per_line=1, symbols=ul)))

<IPython.core.display.Math object>

### $D1V7$

In [293]:
import warnings
warnings.filterwarnings("ignore")
from pylab import *
from sympy import *
import numpy as np
q=7
cx=np.array([0,1,-1,2,-2,3,-3])
wis=np.array([w[0],w[1],w[2],w[3],w[4],w[5],w[6]])
# w0 = sp.Rational(16, 45) + sp.Rational(4, 45)*sp.sqrt(10)
# w1 = sp.Rational(3, 10) - sp.Rational(3, 80)*sp.sqrt(10)
# w2 = sp.Rational(1, 45) - sp.Rational(1, 144)*sp.sqrt(10)
feq = us*wis

In [294]:
from itertools import combinations
import sympy as sp

OrderT = sp.Matrix([
    sum(
        (-omg1)**(n-1)
        * sp.prod(Txc[i] for i in idx)
        * (omega * sum(feq[i] for i in idx) + (1-omega)*us)
        for idx in combinations(range(q), n)
    )
    for n in range(1, 8)
])

In [296]:
# display(OrderT[0])
mt0=sp.collect(sp.expand(OrderT[0])
           .subs({Txc[1]*Txc[2]:1,Txc[0]:1})
           .subs(Txc[1]*us,ul['ut0d-1'])
           .subs(Txc[2]*us,ul['ut0d+1'])
           .subs(Txc[3]*us,ul['ut0d-2'])
           .subs(Txc[4]*us,ul['ut0d+2'])
           .subs(Txc[5]*us,ul['ut0d-3'])
           .subs(Txc[6]*us,ul['ut0d+3'])    
           # .subs({w[2]:Rational(1,6),w[1]:Rational(1,6),w[0]:Rational(2,3)})
           ,{us,ul['ut0d-1'],ul['ut0d+1'],ul['ut0d-2'],ul['ut0d+2'],ul['ut0d-3'],ul['ut0d+3']}
          )
brokeq2(mt0, terms_per_line=1)

<IPython.core.display.Math object>

In [323]:
# display(OrderT[1])
mt1=sp.collect(sp.expand(OrderT[1])
           .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[5]*Txc[6]:1, Txc[0]:1})
           .subs({Txc[1]*Txc[3]:Txc[5], Txc[2]*Txc[4]:Txc[6], Txc[1]*Txc[4]:Txc[2], Txc[2]*Txc[3]:Txc[1]})
           .subs({Txc[1]*Txc[5]:p2T[(1,5)], Txc[2]*Txc[6]:p2T[(2,6)], Txc[1]*Txc[6]:Txc[4], Txc[2]*Txc[5]:Txc[3]})
           .subs({Txc[3]*Txc[5]:p2T[(3,5)], Txc[4]*Txc[6]:p2T[(4,6)], Txc[3]*Txc[6]:Txc[2], Txc[4]*Txc[5]:Txc[1]})
           .subs(Txc[1]*us,ul['ut-1d-1'])
           .subs(Txc[2]*us,ul['ut-1d+1'])
           .subs(Txc[3]*us,ul['ut-1d-2'])
           .subs(Txc[4]*us,ul['ut-1d+2'])
           .subs(Txc[5]*us,ul['ut-1d-3'])
           .subs(Txc[6]*us,ul['ut-1d+3'])
           .subs(p2T[(1, 5)]*us,ul['ut-1d-4'])
           .subs(p2T[(2, 6)]*us,ul['ut-1d+4'])
           .subs(p2T[(3, 5)]*us,ul['ut-1d-5'])
           .subs(p2T[(4, 6)]*us,ul['ut-1d+5'])
           .subs(us,ul['ut-1'])
           # .subs({w[2]:Rational(1,6),w[1]:Rational(1,6),w[0]:Rational(2,3)})
           ,{ul['ut-1'],ul['ut-1d-1'],ul['ut-1d+1'],ul['ut-1d-2'],ul['ut-1d+2'],ul['ut-1d-3'],ul['ut-1d+3'],ul['ut-1d-4'],ul['ut-1d+4']
            ,ul['ut-1d-5'],ul['ut-1d+5']}
          )
# display(mt1)
brokeq2(mt1, terms_per_line=1)

<IPython.core.display.Math object>

In [335]:
# display(OrderT[2])
mt2=sp.collect(sp.expand(OrderT[2])
           .subs({Txc[1]*Txc[3]*Txc[5]:p3T[(1,3,5)], Txc[2]*Txc[4]*Txc[6]:p3T[(2,4,6)]})
           .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[5]*Txc[6]:1, Txc[0]:1})
           .subs({Txc[1]*Txc[3]:Txc[5], Txc[2]*Txc[4]:Txc[6], Txc[1]*Txc[4]:Txc[2], Txc[2]*Txc[3]:Txc[1]})
           .subs({Txc[1]*Txc[5]:p2T[(1,5)], Txc[2]*Txc[6]:p2T[(2,6)], Txc[1]*Txc[6]:Txc[4], Txc[2]*Txc[5]:Txc[3]})
           .subs({Txc[3]*Txc[5]:p2T[(3,5)], Txc[4]*Txc[6]:p2T[(4,6)], Txc[3]*Txc[6]:Txc[2], Txc[4]*Txc[5]:Txc[1]})
            .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[5]*Txc[6]:1, Txc[0]:1})
           .subs({Txc[1]*Txc[3]:Txc[5], Txc[2]*Txc[4]:Txc[6], Txc[1]*Txc[4]:Txc[2], Txc[2]*Txc[3]:Txc[1]})
           .subs({Txc[1]*Txc[5]:p2T[(1,5)], Txc[2]*Txc[6]:p2T[(2,6)], Txc[1]*Txc[6]:Txc[4], Txc[2]*Txc[5]:Txc[3]})
           .subs({Txc[3]*Txc[5]:p2T[(3,5)], Txc[4]*Txc[6]:p2T[(4,6)], Txc[3]*Txc[6]:Txc[2], Txc[4]*Txc[5]:Txc[1]})
           .subs(Txc[1]*us,ul['ut-2d-1'])
           .subs(Txc[2]*us,ul['ut-2d+1'])
           .subs(Txc[3]*us,ul['ut-2d-2'])
           .subs(Txc[4]*us,ul['ut-2d+2'])
           .subs(Txc[5]*us,ul['ut-2d-3'])
           .subs(Txc[6]*us,ul['ut-2d+3'])
           .subs(p2T[(1, 5)]*us,ul['ut-2d-4'])
           .subs(p2T[(2, 6)]*us,ul['ut-2d+4'])
           .subs(p2T[(3, 5)]*us,ul['ut-2d-5'])
           .subs(p2T[(4, 6)]*us,ul['ut-2d+5'])
           .subs(p3T[(1,3,5)]*us,ul['ut-2d-6'])
           .subs(p3T[(2,4,6)]*us,ul['ut-2d+6'])
           .subs(us,ul['ut-2'])
           # .subs({w[2]:Rational(1,6),w[1]:Rational(1,6),w[0]:Rational(2,3)})
           ,{ul['ut-2'],ul['ut-2d-1'],ul['ut-2d+1'],ul['ut-2d-2'],ul['ut-2d+2'],ul['ut-2d-3'],ul['ut-2d+3'],ul['ut-2d-4'],ul['ut-2d+4']
            ,ul['ut-2d-5'],ul['ut-2d+5'],ul['ut-2d-6'],ul['ut-2d+6']}
          )
# display(mt1)
brokeq2(mt2, terms_per_line=1)

<IPython.core.display.Math object>

In [338]:
# display(OrderT[3])
mt3=sp.collect(sp.expand(OrderT[3])
           .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[5]*Txc[6]:1, Txc[0]:1})
           .subs({Txc[1]*Txc[3]*Txc[5]:p3T[(1,3,5)], Txc[2]*Txc[4]*Txc[6]:p3T[(2,4,6)]})
           .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[5]*Txc[6]:1, Txc[0]:1})
           .subs({Txc[1]*Txc[3]:Txc[5], Txc[2]*Txc[4]:Txc[6], Txc[1]*Txc[4]:Txc[2], Txc[2]*Txc[3]:Txc[1]})
           .subs({Txc[1]*Txc[5]:p2T[(1,5)], Txc[2]*Txc[6]:p2T[(2,6)], Txc[1]*Txc[6]:Txc[4], Txc[2]*Txc[5]:Txc[3]})
           .subs({Txc[3]*Txc[5]:p2T[(3,5)], Txc[4]*Txc[6]:p2T[(4,6)], Txc[3]*Txc[6]:Txc[2], Txc[4]*Txc[5]:Txc[1]})
            .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[5]*Txc[6]:1, Txc[0]:1})
           .subs({Txc[1]*Txc[3]:Txc[5], Txc[2]*Txc[4]:Txc[6], Txc[1]*Txc[4]:Txc[2], Txc[2]*Txc[3]:Txc[1]})
           .subs({Txc[1]*Txc[5]:p2T[(1,5)], Txc[2]*Txc[6]:p2T[(2,6)], Txc[1]*Txc[6]:Txc[4], Txc[2]*Txc[5]:Txc[3]})
           .subs({Txc[3]*Txc[5]:p2T[(3,5)], Txc[4]*Txc[6]:p2T[(4,6)], Txc[3]*Txc[6]:Txc[2], Txc[4]*Txc[5]:Txc[1]})
           .subs(Txc[1]*us,ul['ut-3d-1'])
           .subs(Txc[2]*us,ul['ut-3d+1'])
           .subs(Txc[3]*us,ul['ut-3d-2'])
           .subs(Txc[4]*us,ul['ut-3d+2'])
           .subs(Txc[5]*us,ul['ut-3d-3'])
           .subs(Txc[6]*us,ul['ut-3d+3'])
           .subs(p2T[(1, 5)]*us,ul['ut-3d-4'])
           .subs(p2T[(2, 6)]*us,ul['ut-3d+4'])
           .subs(p2T[(3, 5)]*us,ul['ut-3d-5'])
           .subs(p2T[(4, 6)]*us,ul['ut-3d+5'])
           .subs(p3T[(1,3,5)]*us,ul['ut-3d-6'])
           .subs(p3T[(2,4,6)]*us,ul['ut-3d+6'])
           .subs(us,ul['ut-3'])
           # .subs({w[2]:Rational(1,6),w[1]:Rational(1,6),w[0]:Rational(2,3)})
           ,{ul['ut-3'],ul['ut-3d-1'],ul['ut-3d+1'],ul['ut-3d-2'],ul['ut-3d+2'],ul['ut-3d-3'],ul['ut-3d+3'],ul['ut-3d-4'],ul['ut-3d+4']
            ,ul['ut-3d-5'],ul['ut-3d+5'],ul['ut-3d-6'],ul['ut-3d+6']}
          )
# display(mt3)
brokeq2(mt3, terms_per_line=1)

<IPython.core.display.Math object>

In [339]:
# display(OrderT[4])
mt4=sp.collect(sp.expand(OrderT[4])
           .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[5]*Txc[6]:1, Txc[0]:1})
           .subs({Txc[1]*Txc[3]*Txc[5]:p3T[(1,3,5)], Txc[2]*Txc[4]*Txc[6]:p3T[(2,4,6)]})
           .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[5]*Txc[6]:1, Txc[0]:1})
           .subs({Txc[1]*Txc[3]:Txc[5], Txc[2]*Txc[4]:Txc[6], Txc[1]*Txc[4]:Txc[2], Txc[2]*Txc[3]:Txc[1]})
           .subs({Txc[1]*Txc[5]:p2T[(1,5)], Txc[2]*Txc[6]:p2T[(2,6)], Txc[1]*Txc[6]:Txc[4], Txc[2]*Txc[5]:Txc[3]})
           .subs({Txc[3]*Txc[5]:p2T[(3,5)], Txc[4]*Txc[6]:p2T[(4,6)], Txc[3]*Txc[6]:Txc[2], Txc[4]*Txc[5]:Txc[1]})
            .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[5]*Txc[6]:1, Txc[0]:1})
           .subs({Txc[1]*Txc[3]:Txc[5], Txc[2]*Txc[4]:Txc[6], Txc[1]*Txc[4]:Txc[2], Txc[2]*Txc[3]:Txc[1]})
           .subs({Txc[1]*Txc[5]:p2T[(1,5)], Txc[2]*Txc[6]:p2T[(2,6)], Txc[1]*Txc[6]:Txc[4], Txc[2]*Txc[5]:Txc[3]})
           .subs({Txc[3]*Txc[5]:p2T[(3,5)], Txc[4]*Txc[6]:p2T[(4,6)], Txc[3]*Txc[6]:Txc[2], Txc[4]*Txc[5]:Txc[1]})
           .subs(Txc[1]*us,ul['ut-4d-1'])
           .subs(Txc[2]*us,ul['ut-4d+1'])
           .subs(Txc[3]*us,ul['ut-4d-2'])
           .subs(Txc[4]*us,ul['ut-4d+2'])
           .subs(Txc[5]*us,ul['ut-4d-3'])
           .subs(Txc[6]*us,ul['ut-4d+3'])
           .subs(p2T[(1, 5)]*us,ul['ut-4d-4'])
           .subs(p2T[(2, 6)]*us,ul['ut-4d+4'])
           .subs(p2T[(3, 5)]*us,ul['ut-4d-5'])
           .subs(p2T[(4, 6)]*us,ul['ut-4d+5'])
           .subs(p3T[(1,3,5)]*us,ul['ut-4d-6'])
           .subs(p3T[(2,4,6)]*us,ul['ut-4d+6'])
           .subs(us,ul['ut-4'])
           # .subs({w[2]:Rational(1,6),w[1]:Rational(1,6),w[0]:Rational(2,3)})
           ,{ul['ut-4'],ul['ut-4d-1'],ul['ut-4d+1'],ul['ut-4d-2'],ul['ut-4d+2'],ul['ut-4d-3'],ul['ut-4d+3'],ul['ut-4d-4'],ul['ut-4d+4']
            ,ul['ut-4d-5'],ul['ut-4d+5'],ul['ut-4d-6'],ul['ut-4d+6']}
          )
# display(mt4)
brokeq2(mt4, terms_per_line=1)

<IPython.core.display.Math object>

In [340]:
# display(OrderT[5])
mt5=sp.collect(sp.expand(OrderT[5])
           .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[5]*Txc[6]:1, Txc[0]:1})
           .subs({Txc[1]*Txc[3]*Txc[5]:p3T[(1,3,5)], Txc[2]*Txc[4]*Txc[6]:p3T[(2,4,6)]})
           .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[5]*Txc[6]:1, Txc[0]:1})
           .subs({Txc[1]*Txc[3]:Txc[5], Txc[2]*Txc[4]:Txc[6], Txc[1]*Txc[4]:Txc[2], Txc[2]*Txc[3]:Txc[1]})
           .subs({Txc[1]*Txc[5]:p2T[(1,5)], Txc[2]*Txc[6]:p2T[(2,6)], Txc[1]*Txc[6]:Txc[4], Txc[2]*Txc[5]:Txc[3]})
           .subs({Txc[3]*Txc[5]:p2T[(3,5)], Txc[4]*Txc[6]:p2T[(4,6)], Txc[3]*Txc[6]:Txc[2], Txc[4]*Txc[5]:Txc[1]})
            .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[5]*Txc[6]:1, Txc[0]:1})
           .subs({Txc[1]*Txc[3]:Txc[5], Txc[2]*Txc[4]:Txc[6], Txc[1]*Txc[4]:Txc[2], Txc[2]*Txc[3]:Txc[1]})
           .subs({Txc[1]*Txc[5]:p2T[(1,5)], Txc[2]*Txc[6]:p2T[(2,6)], Txc[1]*Txc[6]:Txc[4], Txc[2]*Txc[5]:Txc[3]})
           .subs({Txc[3]*Txc[5]:p2T[(3,5)], Txc[4]*Txc[6]:p2T[(4,6)], Txc[3]*Txc[6]:Txc[2], Txc[4]*Txc[5]:Txc[1]})
           .subs(Txc[1]*us,ul['ut-5d-1'])
           .subs(Txc[2]*us,ul['ut-5d+1'])
           .subs(Txc[3]*us,ul['ut-5d-2'])
           .subs(Txc[4]*us,ul['ut-5d+2'])
           .subs(Txc[5]*us,ul['ut-5d-3'])
           .subs(Txc[6]*us,ul['ut-5d+3'])
           .subs(p2T[(1, 5)]*us,ul['ut-5d-4'])
           .subs(p2T[(2, 6)]*us,ul['ut-5d+4'])
           .subs(p2T[(3, 5)]*us,ul['ut-5d-5'])
           .subs(p2T[(4, 6)]*us,ul['ut-5d+5'])
           .subs(p3T[(1,3,5)]*us,ul['ut-5d-6'])
           .subs(p3T[(2,4,6)]*us,ul['ut-5d+6'])
           .subs(us,ul['ut-5'])
           # .subs({w[2]:Rational(1,6),w[1]:Rational(1,6),w[0]:Rational(2,3)})
           ,{ul['ut-5'],ul['ut-5d-1'],ul['ut-5d+1'],ul['ut-5d-2'],ul['ut-5d+2'],ul['ut-5d-3'],ul['ut-5d+3'],ul['ut-5d-4'],ul['ut-5d+4']
            ,ul['ut-5d-5'],ul['ut-5d+5'],ul['ut-5d-6'],ul['ut-5d+6']}
          )
# display(mt5)
brokeq2(mt5, terms_per_line=1)

<IPython.core.display.Math object>

In [341]:
# display(OrderT[6])
mt6=sp.collect(sp.expand(OrderT[6])
           .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[5]*Txc[6]:1, Txc[0]:1})
           .subs({Txc[1]*Txc[3]*Txc[5]:p3T[(1,3,5)], Txc[2]*Txc[4]*Txc[6]:p3T[(2,4,6)]})
           .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[5]*Txc[6]:1, Txc[0]:1})
           .subs({Txc[1]*Txc[3]:Txc[5], Txc[2]*Txc[4]:Txc[6], Txc[1]*Txc[4]:Txc[2], Txc[2]*Txc[3]:Txc[1]})
           .subs({Txc[1]*Txc[5]:p2T[(1,5)], Txc[2]*Txc[6]:p2T[(2,6)], Txc[1]*Txc[6]:Txc[4], Txc[2]*Txc[5]:Txc[3]})
           .subs({Txc[3]*Txc[5]:p2T[(3,5)], Txc[4]*Txc[6]:p2T[(4,6)], Txc[3]*Txc[6]:Txc[2], Txc[4]*Txc[5]:Txc[1]})
            .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[5]*Txc[6]:1, Txc[0]:1})
           .subs({Txc[1]*Txc[3]:Txc[5], Txc[2]*Txc[4]:Txc[6], Txc[1]*Txc[4]:Txc[2], Txc[2]*Txc[3]:Txc[1]})
           .subs({Txc[1]*Txc[5]:p2T[(1,5)], Txc[2]*Txc[6]:p2T[(2,6)], Txc[1]*Txc[6]:Txc[4], Txc[2]*Txc[5]:Txc[3]})
           .subs({Txc[3]*Txc[5]:p2T[(3,5)], Txc[4]*Txc[6]:p2T[(4,6)], Txc[3]*Txc[6]:Txc[2], Txc[4]*Txc[5]:Txc[1]})
           .subs(Txc[1]*us,ul['ut-6d-1'])
           .subs(Txc[2]*us,ul['ut-6d+1'])
           .subs(Txc[3]*us,ul['ut-6d-2'])
           .subs(Txc[4]*us,ul['ut-6d+2'])
           .subs(Txc[5]*us,ul['ut-6d-3'])
           .subs(Txc[6]*us,ul['ut-6d+3'])
           .subs(p2T[(1, 5)]*us,ul['ut-6d-4'])
           .subs(p2T[(2, 6)]*us,ul['ut-6d+4'])
           .subs(p2T[(3, 5)]*us,ul['ut-6d-5'])
           .subs(p2T[(4, 6)]*us,ul['ut-6d+5'])
           .subs(p3T[(1,3,5)]*us,ul['ut-6d-6'])
           .subs(p3T[(2,4,6)]*us,ul['ut-6d+6'])
           .subs(us,ul['ut-6'])
           # .subs({w[2]:Rational(1,6),w[1]:Rational(1,6),w[0]:Rational(2,3)})
           ,{ul['ut-6'],ul['ut-6d-1'],ul['ut-6d+1'],ul['ut-6d-2'],ul['ut-6d+2'],ul['ut-6d-3'],ul['ut-6d+3'],ul['ut-6d-4'],ul['ut-6d+4']
            ,ul['ut-6d-5'],ul['ut-6d+5'],ul['ut-6d-6'],ul['ut-6d+6']}
          )
# display(mt6)
brokeq2(mt6, terms_per_line=1)

<IPython.core.display.Math object>

In [342]:
display(Math(fdm_ordered_brokeq(sp.Eq(ul['ut+1'],mt0+mt1+mt2+mt3+mt4+mt5+mt6), terms_per_line=1, symbols=ul)))

<IPython.core.display.Math object>

 ## Two-Dimensional Cases: $D2Qq$ Lattices

In [348]:
import sympy as sp
uly = {}
past_steps = [1, 2, 3, 4, 5, 6,]
space_offsets = [1, 2, 3, 4, 5, 6]
# time_map: key_prefix -> (latex_expr): "+1" -> u^{t+1},  "0" -> u^{t},  "-1".."-4" -> u^{t-k}
time_map = {"+1": "t+1","0":  "t",**{f"-{k}": f"t-{k}" for k in past_steps},}

# Non-shifted symbols
for tk, te in time_map.items():
    uly[f"ut{tk}"] = sp.Symbol(fr'u^{{{te}}}_{{y}}')

# Shifted symbols (skip the future step "+1")
for tk, te in time_map.items():
    if tk == "+1":
        continue
    for ssign in ("+", "-"):
        for d in space_offsets:
            uly[f"ut{tk}d{ssign}{d}"] = sp.Symbol(fr'u^{{{te}}}_{{y{ssign}{d}}}')

In [349]:
uly

{'ut+1': u^{t+1}_{y},
 'ut0': u^{t}_{y},
 'ut-1': u^{t-1}_{y},
 'ut-2': u^{t-2}_{y},
 'ut-3': u^{t-3}_{y},
 'ut-4': u^{t-4}_{y},
 'ut-5': u^{t-5}_{y},
 'ut-6': u^{t-6}_{y},
 'ut0d+1': u^{t}_{y+1},
 'ut0d+2': u^{t}_{y+2},
 'ut0d+3': u^{t}_{y+3},
 'ut0d+4': u^{t}_{y+4},
 'ut0d+5': u^{t}_{y+5},
 'ut0d+6': u^{t}_{y+6},
 'ut0d-1': u^{t}_{y-1},
 'ut0d-2': u^{t}_{y-2},
 'ut0d-3': u^{t}_{y-3},
 'ut0d-4': u^{t}_{y-4},
 'ut0d-5': u^{t}_{y-5},
 'ut0d-6': u^{t}_{y-6},
 'ut-1d+1': u^{t-1}_{y+1},
 'ut-1d+2': u^{t-1}_{y+2},
 'ut-1d+3': u^{t-1}_{y+3},
 'ut-1d+4': u^{t-1}_{y+4},
 'ut-1d+5': u^{t-1}_{y+5},
 'ut-1d+6': u^{t-1}_{y+6},
 'ut-1d-1': u^{t-1}_{y-1},
 'ut-1d-2': u^{t-1}_{y-2},
 'ut-1d-3': u^{t-1}_{y-3},
 'ut-1d-4': u^{t-1}_{y-4},
 'ut-1d-5': u^{t-1}_{y-5},
 'ut-1d-6': u^{t-1}_{y-6},
 'ut-2d+1': u^{t-2}_{y+1},
 'ut-2d+2': u^{t-2}_{y+2},
 'ut-2d+3': u^{t-2}_{y+3},
 'ut-2d+4': u^{t-2}_{y+4},
 'ut-2d+5': u^{t-2}_{y+5},
 'ut-2d+6': u^{t-2}_{y+6},
 'ut-2d-1': u^{t-2}_{y-1},
 'ut-2d-2': u^{t-2}_{y-2},

### $D2Q5$

In [344]:
import warnings
warnings.filterwarnings("ignore")
from pylab import *
from sympy import *
import numpy as np
q=5
cx=np.array([0,1,-1,0,0])
cy=np.array([0,0,0,1,-2])
wis=np.array([w[0],w[1],w[2],w[1],w[2]])
feq = us*wis

In [345]:
from itertools import combinations
import sympy as sp

OrderT = sp.Matrix([
    sum(
        (-omg1)**(n-1)
        * sp.prod(Txc[i] for i in idx)
        * (omega * sum(feq[i] for i in idx) + (1-omega)*us)
        for idx in combinations(range(q), n)
    )
    for n in range(1, 6)
])

In [ ]:
# display(OrderT[0])
mt0=sp.collect(OrderT[0]
           .subs({Txc[1]*Txc[2]:1,Txc[0]:1})
           .subs(Txc[1]*us,ul['ut0d-1'])
           .subs(Txc[2]*us,ul['ut0d+1'])
           .subs(Txc[3]*us,ul['ut0d-3'])
           .subs(Txc[4]*us,ul['ut0d+3'])
           # .subs({w[2]:Rational(1,6),w[1]:Rational(1,6),w[0]:Rational(2,3)})
           ,{us,ul['ut0d-1'],ul['ut0d+1'],ul['ut0d-3'],ul['ut0d+3']}
          )
display(mt0)